# AutoQual AI+ — Organization-Y Security Event Classifier

Trains **Model 3 — Security Sequence Prediction (event classification)** on the
`organization-y` archive: real multi-source server logs (Apache access/error logs across
~15 domains, `auth.log`/`syslog`, and CyberPanel admin-panel logs) from a documented server
compromise, labeled via a YAML rule engine (`ground-truth/organization-y.yaml`).

Unlike the CSIC notebook (one flat pre-built CSV), this dataset is a **zip of raw,
heterogeneous log files** — so this notebook has two parts:

- **Part A** — walk every file in the archive, parse each log line with a per-format parser,
  label it by running it through the ground-truth YAML rules, and stream everything into a
  single normalized CSV (`organization_y_events.csv`).
- **Part B** — the same rigor as the CSIC notebook: EDA → feature engineering →
  preprocessing pipeline → baseline model comparison → hyperparameter tuning → final
  evaluation → save artifact for deployment.

**Note on scale:** the raw archive is ~4.5GB uncompressed across ~200 files (one single
file, `syslog.1`, is ~2.4GB / ~7M lines on its own). Part A streams every file line-by-line
(never loads a whole file into memory) and keeps memory flat regardless of file size — but
it still has to *read* every line to classify it, so the first run will take a while
(realistically tens of minutes). Consider using "Save & Run All" rather than running
interactively.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


In [ ]:
# Core imports for this notebook
import re
import csv
import gzip
import json
import time
import random
import zipfile
import warnings
from collections import Counter
import importlib.metadata as importlib_metadata

import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
RANDOM_STATE = 42


## Part A — Convert the raw log archive into a labeled CSV

### 1. Locate the dataset root and the ground-truth rules file

Kaggle auto-extracts zipped datasets, but the exact mount path depends on how the dataset
was uploaded/named — rather than hardcode a guess, this searches `/kaggle/input` for the
anchor file `organization-y.yaml` (the ground-truth rules) and derives everything else from
its location. If a raw `.zip` is found instead of an already-extracted folder, it's
extracted to `/kaggle/working/` first (since `/kaggle/input` is read-only).


In [ ]:
KAGGLE_INPUT_ROOT = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

def find_ground_truth_yaml(search_root):
    for dirpath, _, filenames in os.walk(search_root):
        for fname in filenames:
            if fname == 'organization-y.yaml':
                return os.path.join(dirpath, fname)
    return None

def find_dataset_zip(search_root):
    for dirpath, _, filenames in os.walk(search_root):
        for fname in filenames:
            if fname.lower() == 'organization-y.zip':
                return os.path.join(dirpath, fname)
    return None

yaml_path = find_ground_truth_yaml(KAGGLE_INPUT_ROOT)

if yaml_path is None:
    zip_path = find_dataset_zip(KAGGLE_INPUT_ROOT)
    if zip_path is None:
        raise FileNotFoundError(
            'Could not find organization-y.yaml or organization-y.zip under /kaggle/input. '
            'Make sure the organization-y dataset is attached to this notebook.'
        )
    print('Found zip at', zip_path, '- extracting to', WORKING_DIR)
    extract_dir = os.path.join(WORKING_DIR, 'organization-y-extracted')
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    yaml_path = find_ground_truth_yaml(extract_dir)
    if yaml_path is None:
        raise FileNotFoundError('Extracted the zip but still could not find organization-y.yaml inside it.')

DATASET_ROOT = os.path.dirname(os.path.dirname(yaml_path))  # .../organization-y (parent of ground-truth/)
LOG_ROOT = os.path.join(DATASET_ROOT, 'log')

print('Ground-truth YAML:', yaml_path)
print('Dataset root:', DATASET_ROOT)
print('Log root:', LOG_ROOT)
assert os.path.isdir(LOG_ROOT), f'Expected a log/ directory under {DATASET_ROOT}, did not find one.'


### 2. Per-format log line parsers

Four real formats appear in this archive, each parsed by its own regex (validated against real sample lines from this exact dataset before being placed here); anything that doesn't match any of them falls back to a generic parser that just keeps the raw line as `message` — so nothing crashes on an unfamiliar line, it just loses structured fields (timestamp/IP) for that one line.

In [ ]:
ACCESS_LOG_RE = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ \[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<path>\S+) (?P<protocol>[^"]+)" '
    r'(?P<status>\d+) (?P<size>\S+) "(?P<referer>[^"]*)" "(?P<user_agent>[^"]*)"'
)

DOMAIN_ERROR_LOG_RE = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+) '
    r'\[(?P<level>[A-Z]+)\] \[(?P<pid>\d+)\] \[(?P<client>[^\]]+)\] (?P<message>.*)$'
)

SYSLOG_RE = re.compile(
    r'^(?P<timestamp>\w{3}\s+\d{1,2} \d{2}:\d{2}:\d{2}) (?P<host>\S+) '
    r'(?P<process>[^\[:]+)(\[(?P<pid>\d+)\])?: (?P<message>.*)$'
)

CYBERPANEL_LOG_RE = re.compile(
    r'^\[(?P<timestamp>[\d._-]+)\] (?P<message>.*)$'
)

IP_IN_MESSAGE_RE = re.compile(r'\b\d{1,3}(?:\.\d{1,3}){3}\b')


def parse_access_log_line(line):
    m = ACCESS_LOG_RE.match(line)
    if not m:
        return None
    d = m.groupdict()
    return {
        'timestamp': d['timestamp'], 'client_ip': d['ip'],
        'message': f"{d['method']} {d['path']} {d['protocol']} {d['status']} {d['referer']} {d['user_agent']}",
        'status': d['status'], 'method': d['method'], 'path': d['path'], 'user_agent': d['user_agent'],
    }

def parse_domain_error_log_line(line):
    m = DOMAIN_ERROR_LOG_RE.match(line)
    if not m:
        return None
    d = m.groupdict()
    client_ip = d['client'].split(':')[0]
    return {'timestamp': d['timestamp'], 'client_ip': client_ip, 'message': d['message']}

def parse_syslog_line(line):
    m = SYSLOG_RE.match(line)
    if not m:
        return None
    d = m.groupdict()
    ip_match = IP_IN_MESSAGE_RE.search(d['message'])
    return {
        'timestamp': d['timestamp'], 'client_ip': ip_match.group(0) if ip_match else '',
        'message': f"{d['process'].strip()}: {d['message']}",
    }

def parse_cyberpanel_log_line(line):
    m = CYBERPANEL_LOG_RE.match(line)
    if not m:
        return None
    d = m.groupdict()
    ip_match = IP_IN_MESSAGE_RE.search(d['message'])
    return {'timestamp': d['timestamp'], 'client_ip': ip_match.group(0) if ip_match else '', 'message': d['message']}

def parse_generic_line(line):
    ip_match = IP_IN_MESSAGE_RE.search(line)
    return {'timestamp': '', 'client_ip': ip_match.group(0) if ip_match else '', 'message': line}


PARSERS_BY_LOGTYPE = {
    'access_log': parse_access_log_line,
    'domain_error_log': parse_domain_error_log_line,
    'syslog': parse_syslog_line,
    'cyberpanel_log': parse_cyberpanel_log_line,
    'generic': parse_generic_line,
}

def parse_line(line, log_type):
    parser = PARSERS_BY_LOGTYPE.get(log_type, parse_generic_line)
    result = parser(line)
    if result is None:
        result = parse_generic_line(line)
    return result

### 3. Load the ground-truth labeling rules

Note the `%pattern%` stripping below — several rules (e.g. `bruteforce_login_web`) use SQL-LIKE-style wildcard wrapping where `%` just means "contains", not a literal character to search for. Missing this turns those rules into permanent no-ops, so it's handled explicitly.

In [ ]:
def load_rules(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        raw_rules = yaml.safe_load(f)
    rules = []
    for r in raw_rules:
        # Some rules use SQL-LIKE-style "%pattern%" wildcard wrapping (e.g. bruteforce_login_web);
        # '%' here just marks "contains" and isn't a literal character to search for, so strip it.
        rules.append({
            'label': r['ground_truth_label'],
            'patterns': [p.lower().replace('%', '') for p in r['filter']],
        })
    return rules

def classify_line(message, rules):
    msg_lower = message.lower()
    for rule in rules:
        if all(p in msg_lower for p in rule['patterns']):
            return rule['label']
    return 'benign'

RULES = load_rules(yaml_path)
print(f'Loaded {len(RULES)} rules, {len(set(r["label"] for r in RULES))} distinct labels:')
print(sorted(set(r['label'] for r in RULES)))


### 4. Route every file to the right parser

Filename-pattern based; anything unrecognized safely falls through to `generic` (see the
parser fallback above), so an unexpected file never crashes the run — it just contributes
unstructured rows instead of nicely-parsed ones.


In [ ]:
def classify_file(path):
    lower = path.lower()
    if 'access_log' in lower or 'access.log' in lower:
        return 'access_log'
    if 'auth.log' in lower or 'syslog' in lower:
        return 'syslog'
    if 'error-logs.txt' in lower:
        return 'cyberpanel_log'
    if 'error_log' in lower or 'error.log' in lower:
        return 'domain_error_log'  # falls back to generic automatically if the format doesn't match
    return 'generic'

all_files = []
for dirpath, _, filenames in os.walk(LOG_ROOT):
    for fname in filenames:
        full_path = os.path.join(dirpath, fname)
        all_files.append((full_path, classify_file(full_path)))

print(f'Found {len(all_files)} files to process')
log_type_counts = Counter(lt for _, lt in all_files)
for lt, count in log_type_counts.most_common():
    print(f'  {lt}: {count} files')


### 5. Stream-parse every file into one labeled CSV

Every line is parsed and classified, but **only non-benign lines plus a random sample of
benign lines are kept** (`BENIGN_KEEP_RATE` below) — the archive is overwhelmingly benign
background noise (normal traffic, routine cron/systemd chatter), and keeping 100% of it
would make the resulting CSV huge and the downstream training step painfully slow for
little added signal. All rule-matched (non-benign) lines are always kept in full.

Tune `BENIGN_KEEP_RATE` up if you want a larger benign sample for more robust negative
examples, or down if the output CSV is too large for your session.


In [ ]:
OUTPUT_CSV = os.path.join(WORKING_DIR, 'organization_y_events.csv')
BENIGN_KEEP_RATE = 0.02  # keep 2% of non-matching ('benign') lines
PROGRESS_EVERY_FILES = 10

random.seed(RANDOM_STATE)

def open_log_file(path):
    if path.endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8', errors='replace')
    return open(path, 'r', encoding='utf-8', errors='replace')

start_time = time.time()
total_lines_seen = 0
total_rows_kept = 0
label_counter = Counter()
files_with_errors = []

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as out_f:
    writer = csv.writer(out_f)
    writer.writerow(['source_file', 'log_type', 'timestamp', 'client_ip', 'message', 'label'])

    for i, (path, log_type) in enumerate(all_files, start=1):
        rel_path = os.path.relpath(path, LOG_ROOT)
        try:
            with open_log_file(path) as f:
                for line in f:
                    line = line.rstrip('\n')
                    if not line:
                        continue
                    total_lines_seen += 1
                    parsed = parse_line(line, log_type)
                    label = classify_line(parsed['message'], RULES)
                    label_counter[label] += 1

                    if label == 'benign' and random.random() > BENIGN_KEEP_RATE:
                        continue

                    writer.writerow([rel_path, log_type, parsed['timestamp'], parsed['client_ip'], parsed['message'], label])
                    total_rows_kept += 1
        except Exception as e:
            files_with_errors.append((rel_path, str(e)))
            continue

        if i % PROGRESS_EVERY_FILES == 0 or i == len(all_files):
            elapsed = time.time() - start_time
            print(f'[{i}/{len(all_files)}] files done | {total_lines_seen:,} lines seen | '
                  f'{total_rows_kept:,} rows kept | {elapsed:.0f}s elapsed')

elapsed_total = time.time() - start_time
print()
print(f'Done in {elapsed_total:.0f}s. {total_lines_seen:,} lines seen, {total_rows_kept:,} rows kept.')
if files_with_errors:
    print(f'{len(files_with_errors)} files hit an error and were skipped (first 5):')
    for rel_path, err in files_with_errors[:5]:
        print(f'  {rel_path}: {err}')


In [ ]:
print('Label distribution across ALL lines seen (before benign downsampling):')
for label, count in label_counter.most_common():
    print(f'  {label}: {count:,}')


## Part B — Train a classifier on the converted CSV

Same structure/rigor as the CSIC notebook: EDA → feature engineering → preprocessing
pipeline → baseline comparison → tuning → final evaluation → save artifact. This is now a
**multi-class** problem (every `ground_truth_label` plus `benign`), not binary.


### 6. Load the converted CSV

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print('Shape:', df.shape)
df.head()


### 7. Exploratory Data Analysis

In [ ]:
print('Label distribution (kept rows):')
print(df['label'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['label'].value_counts().plot(kind='barh', ax=axes[0])
axes[0].set_title('Label distribution (kept rows)')
axes[0].invert_yaxis()

df['log_type'].value_counts().plot(kind='bar', ax=axes[1], color='#2a6f9a')
axes[1].set_title('Source log type distribution')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()


### 8. Feature Engineering

In [ ]:
SUSPICIOUS_KEYWORDS = [
    'select', 'union select', '<script', 'javascript:', '../', 'etc/passwd',
    'cmd.exe', '/bin/sh', 'failed password', 'invalid user', 'authentication failure', '[sudo]'
]

def count_suspicious_keywords(text):
    text_lower = text.lower()
    return sum(text_lower.count(kw) for kw in SUSPICIOUS_KEYWORDS)

df['message'] = df['message'].fillna('')
df['log_type'] = df['log_type'].fillna('unknown')

df['message_len'] = df['message'].str.len()
df['digit_count'] = df['message'].apply(lambda t: sum(ch.isdigit() for ch in t))
df['special_char_count'] = df['message'].apply(lambda t: sum(t.count(c) for c in ['<', '>', chr(39), chr(34), ';', '%']))
df['suspicious_keyword_count'] = df['message'].apply(count_suspicious_keywords)
df['has_client_ip'] = df['client_ip'].notna().astype(int)

df[['message_len', 'digit_count', 'special_char_count', 'suspicious_keyword_count']].describe()


In [ ]:
# Drop labels with too few examples to reliably stratify-split or learn from
MIN_EXAMPLES_PER_LABEL = 5
label_counts = df['label'].value_counts()
rare_labels = label_counts[label_counts < MIN_EXAMPLES_PER_LABEL].index.tolist()
if rare_labels:
    print(f'Dropping {len(rare_labels)} labels with < {MIN_EXAMPLES_PER_LABEL} examples:')
    for lbl in rare_labels:
        print(f'  {lbl}: {label_counts[lbl]} examples')
    df = df[~df['label'].isin(rare_labels)].reset_index(drop=True)
print('Shape after rare-label filter:', df.shape)


### 9. Preprocessing Pipeline

In [ ]:
NUMERIC_FEATURES = ['message_len', 'digit_count', 'special_char_count', 'suspicious_keyword_count', 'has_client_ip']
CATEGORICAL_FEATURES = ['log_type']
TEXT_FEATURE = 'message'

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ('text', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2), TEXT_FEATURE),
])


### 10. Train / Test Split

Labels are integer-encoded here (not left as strings). LogisticRegression/RandomForest
would happily fit on the raw string labels, but XGBoost's sklearn wrapper requires
`y` to be integers `0..n_classes-1` and raises `ValueError: Invalid classes inferred...`
otherwise — encoding once up front keeps every model in the comparison on equal footing.


In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])
CLASS_NAMES = label_encoder.classes_  # CLASS_NAMES[i] is the string label for encoded class i

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Classes:', list(CLASS_NAMES))


### 11. Baseline Model Comparison

In [ ]:
candidate_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1),
}

results = []

for name, clf in candidate_models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision_weighted': precision,
        'recall_weighted': recall,
        'f1_weighted': f1,
    })

    print(f'=== {name} ===')
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))

results_df = pd.DataFrame(results).sort_values('f1_weighted', ascending=False).reset_index(drop=True)
results_df


In [ ]:
results_df.set_index('model')[['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']].plot(
    kind='bar', figsize=(10, 5), ylim=(0, 1)
)
plt.title('Baseline model comparison')
plt.ylabel('score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

BEST_MODEL_NAME = results_df.iloc[0]['model']
print('Best baseline model by weighted F1:', BEST_MODEL_NAME)


### 12. Hyperparameter Tuning

Tunes whichever model won the baseline comparison, optimizing weighted F1 (more meaningful
than accuracy on an imbalanced multi-class problem like this one).

**Why this searches on a subsample, not the full training set:** `RandomizedSearchCV`
with `n_jobs > 1` runs folds/candidates in separate worker *processes*, and each one holds
its own full copy of the TF-IDF-fitted training data. On a small dataset (like the CSIC
notebook's ~85k rows) that's fine; on this dataset — sourced from multi-GB raw server logs —
`n_jobs=-1` × several worker copies of the whole training set is exactly what exceeds a
Kaggle session's RAM and gets a worker `SIGKILL`'d (`TerminatedWorkerError`). Searching on a
bounded, stratified subsample keeps peak memory constant regardless of how large the full
dataset ends up being, and the winning hyperparameters are then refit on the *full* training
set afterward — so the final model still learns from everything, only the search itself is
capped.


In [ ]:
PARAM_GRIDS = {
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__penalty': ['l2'],
    },
    'RandomForest': {
        'clf__n_estimators': [150, 250, 400],
        'clf__max_depth': [None, 10, 20, 40],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
    },
    'XGBoost': {
        'clf__n_estimators': [150, 250, 400],
        'clf__max_depth': [3, 5, 7, 9],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'clf__subsample': [0.7, 0.85, 1.0],
        'clf__colsample_bytree': [0.7, 0.85, 1.0],
    },
}

def make_tuning_estimator(model_name):
    # Fresh instance with n_jobs pinned to 1 for the inner model — RandomizedSearchCV
    # below already parallelizes across folds/candidates; nesting a second parallel
    # pool inside each of those workers multiplies memory use further on top of the
    # per-worker data copies described above.
    if model_name == 'LogisticRegression':
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    if model_name == 'RandomForest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
    return XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=1)

# Bounded, stratified subsample for the search step only (see markdown above).
SEARCH_SAMPLE_SIZE = 60_000
if len(X_train) > SEARCH_SAMPLE_SIZE:
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=SEARCH_SAMPLE_SIZE, random_state=RANDOM_STATE, stratify=y_train
    )
    print(f'Searching on a stratified subsample: {len(X_search)} of {len(X_train)} training rows')
else:
    X_search, y_search = X_train, y_train
    print(f'Training set ({len(X_train)} rows) is already <= {SEARCH_SAMPLE_SIZE} — searching on all of it')

base_pipeline = Pipeline([('preprocess', preprocessor), ('clf', make_tuning_estimator(BEST_MODEL_NAME))])

search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=PARAM_GRIDS[BEST_MODEL_NAME],
    n_iter=15,
    scoring='f1_weighted',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=2,  # bounded, not -1 — each worker holds its own data copy, see markdown above
    verbose=1,
)
search.fit(X_search, y_search)

print('Best CV weighted-F1 (on search subsample):', search.best_score_)
print('Best params:', search.best_params_)

# Refit the winning hyperparameters on the FULL training set — the subsample above
# was only for finding good params cheaply, not for the final model itself.
best_pipeline = Pipeline([('preprocess', preprocessor), ('clf', make_tuning_estimator(BEST_MODEL_NAME))])
best_pipeline.set_params(**search.best_params_)
best_pipeline.fit(X_train, y_train)
print('Refit best pipeline on the full training set:', X_train.shape)


### 13. Final Evaluation (tuned model, held-out test set)

In [ ]:
final_preds = best_pipeline.predict(X_test)
print(classification_report(y_test, final_preds, target_names=CLASS_NAMES, zero_division=0))
final_f1_weighted = f1_score(y_test, final_preds, average='weighted', zero_division=0)
print('Final weighted F1:', final_f1_weighted)


In [ ]:
cm = confusion_matrix(y_test, final_preds, labels=range(len(CLASS_NAMES)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### 14. Save Model Artifact for Deployment

Same handoff contract as the CSIC notebook: the full pipeline (preprocessing + tuned model)
plus a metadata file with the input schema, label set, and library versions. The
`label_encoder` is saved separately too — predictions from the pipeline come back as
integers (`0..n_classes-1`), and the encoder is what maps them back to label strings
like `"bruteforce_login_web"` in the FastAPI serving layer.


In [ ]:
MODEL_OUT_PATH = os.path.join(WORKING_DIR, 'organization_y_event_classifier.pkl')
LABEL_ENCODER_OUT_PATH = os.path.join(WORKING_DIR, 'organization_y_label_encoder.pkl')
METADATA_OUT_PATH = os.path.join(WORKING_DIR, 'organization_y_model_metadata.json')

joblib.dump(best_pipeline, MODEL_OUT_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_OUT_PATH)

def get_version(pkg):
    try:
        return importlib_metadata.version(pkg)
    except importlib_metadata.PackageNotFoundError:
        return None

metadata = {
    'model_name': 'organization_y_security_event_classifier',
    'best_model_type': BEST_MODEL_NAME,
    'best_params': search.best_params_,
    'classes': CLASS_NAMES.tolist(),
    'class_index_to_label': {int(i): name for i, name in enumerate(CLASS_NAMES)},
    'benign_label': 'benign',
    'raw_output_columns': ['source_file', 'log_type', 'timestamp', 'client_ip', 'message', 'label'],
    'note': 'Pass raw log lines through parse_line() + engineer the message_len/digit_count/'
            'special_char_count/suspicious_keyword_count/has_client_ip features above before '
            'calling pipeline.predict() / predict_proba() — the pipeline itself only handles '
            'the ColumnTransformer stage, not the raw-line parsing or feature-derivation steps. '
            'predict() returns an integer class index — decode it with label_encoder.pkl '
            '(or class_index_to_label above) to get the label string back.',
    'benign_keep_rate_used': BENIGN_KEEP_RATE,
    'min_examples_per_label_used': MIN_EXAMPLES_PER_LABEL,
    'test_metrics': {
        'accuracy': accuracy_score(y_test, final_preds),
        'f1_weighted': final_f1_weighted,
    },
    'library_versions': {
        'python': __import__('sys').version,
        'scikit-learn': get_version('scikit-learn'),
        'pandas': get_version('pandas'),
        'numpy': get_version('numpy'),
        'xgboost': get_version('xgboost'),
        'joblib': get_version('joblib'),
        'pyyaml': get_version('pyyaml'),
    },
}

with open(METADATA_OUT_PATH, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Saved model to:', MODEL_OUT_PATH)
print('Saved label encoder to:', LABEL_ENCODER_OUT_PATH)
print('Saved metadata to:', METADATA_OUT_PATH)
print()
print(json.dumps(metadata, indent=2, default=str))


### Handoff checklist
Download all four from `/kaggle/working/`:
- `organization_y_event_classifier.pkl` — the trained pipeline
- `organization_y_label_encoder.pkl` — decodes predicted integer classes back to label strings
- `organization_y_model_metadata.json` — schema, label set, params, versions
- `organization_y_events.csv` — the converted dataset itself (useful to keep for retraining later)

The `parse_line()` / `classify_file()` / feature-engineering functions above also need to
travel with the model — same as the CSIC notebook, the saved pipeline only covers the
`ColumnTransformer` stage, not the raw-log → structured-row step before it.
